In [ ]:
# 1. Environment
import os, shutil, subprocess, sys
import torch
print('torch', torch.__version__)
print('gpus', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(' gpu', i, torch.cuda.get_device_name(i),
          round(torch.cuda.get_device_properties(i).total_memory/1024**3,1), 'GB')
print('working free GB', round(shutil.disk_usage('/kaggle/working').free/1024**3,1))
# /kaggle/temp does not exist on every image; create it and fall back.
TEMP = '/kaggle/temp'
try:
    os.makedirs(TEMP, exist_ok=True)
    print('temp free GB', round(shutil.disk_usage(TEMP).free/1024**3,1))
except OSError:
    TEMP = '/kaggle/working/temp'
    os.makedirs(TEMP, exist_ok=True)
    print('temp fallback', TEMP)
print('TEMP =', TEMP)

In [ ]:
# 2. Which beats to render
BEATS = ['c01', 'c02', 'c10', 'c11', 'c15', 'c31', 'c37']
print('beats:', BEATS or 'all')

In [ ]:
# 3. Clone the pipeline
import os, subprocess
REPO = '/kaggle/working/repo'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','--depth','1','https://github.com/dudefaiz09boop/wan22-production-pipeline.git',REPO], check=True)
os.chdir(REPO)
print(subprocess.run(['git','log','--oneline','-1'],capture_output=True,text=True).stdout)

In [ ]:
# 4. ComfyUI + GGUF node
# Weights go to /kaggle/temp so they do not count against the 20 GB
# working quota; /kaggle/working only holds the repo and the outputs.
import os, subprocess
COMFY = os.path.join(TEMP, 'ComfyUI')
if not os.path.isdir(COMFY):
    subprocess.run(['git','clone','--depth','1',
        'https://github.com/comfyanonymous/ComfyUI.git', COMFY], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','-r',
        f'{COMFY}/requirements.txt'], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q',
        'gguf','sentencepiece','protobuf'], check=True)
    subprocess.run(['git','clone','--depth','1',
        'https://github.com/city96/ComfyUI-GGUF.git',
        f'{COMFY}/custom_nodes/ComfyUI-GGUF'], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','-r',
        f'{COMFY}/custom_nodes/ComfyUI-GGUF/requirements.txt'], check=True)
print('ComfyUI ready at', COMFY)

In [ ]:
# 5. Download the Qwen-Image 2.1 GGUF weights
import os
from huggingface_hub import hf_hub_download

COMFY = os.path.join(TEMP, 'ComfyUI')
DIRS = {
    'diffusion_models': f'{COMFY}/models/diffusion_models',
    'text_encoders': f'{COMFY}/models/text_encoders',
    'vae': f'{COMFY}/models/vae',
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

WANT = [
    ('city96/Qwen-Image-gguf', 'qwen-image-Q3_K_S.gguf', 'diffusion_models'),
    ('Comfy-Org/Qwen-Image_ComfyUI',
     'split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors',
     'text_encoders'),
    ('Comfy-Org/Qwen-Image_ComfyUI',
     'split_files/vae/qwen_image_vae.safetensors', 'vae'),
]
for repo, filename, kind in WANT:
    dest = DIRS[kind]
    target = os.path.join(dest, os.path.basename(filename))
    if os.path.isfile(target) and os.path.getsize(target) > 1_000_000:
        print('cached', target); continue
    path = hf_hub_download(repo_id=repo, filename=filename, local_dir=dest)
    if path != target:
        os.replace(path, target)
    print('downloaded', target, round(os.path.getsize(target)/1024**3,1), 'GB')

In [ ]:
# 6. Start one ComfyUI per GPU
# ComfyUI cannot shard a single model across devices, but two instances
# can render different beats at once, which halves the wall time.
import os, subprocess, time, urllib.request
COMFY = os.path.join(TEMP, 'ComfyUI')
PORTS = [8188, 8189]
procs = []
for index, port in enumerate(PORTS):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(index))
    log = open(os.path.join(TEMP, f'comfy{port}.log'), 'w')
    procs.append(subprocess.Popen(
        [sys.executable, 'main.py', '--listen', '127.0.0.1',
         '--port', str(port), '--lowvram'],
        cwd=COMFY, stdout=log, stderr=subprocess.STDOUT, env=env))
    print('started ComfyUI on GPU', index, 'port', port)

for port in PORTS:
    for _ in range(180):
        try:
            urllib.request.urlopen(f'http://127.0.0.1:{port}/system_stats', timeout=2)
            print('port', port, 'up'); break
        except Exception:
            time.sleep(2)
    else:
        print(open(os.path.join(TEMP, f'comfy{port}.log')).read()[-2000:])
        raise SystemExit(f'ComfyUI on {port} did not start')

In [ ]:
# 7. Render the startframes with Qwen-Image 2.1
import subprocess, sys
servers = ','.join(f'http://127.0.0.1:{p}' for p in PORTS)
cmd = [sys.executable, 'scripts/generate_stills_qwen.py',
       '--output', 'startframes', '--servers', servers]
if BEATS:
    cmd += ['--beats', *BEATS]
print('RUN:', ' '.join(cmd), flush=True)
rc = subprocess.call(cmd, stdout=sys.stdout, stderr=sys.stderr)
print('rc:', rc, flush=True)
import glob
made = sorted(glob.glob('startframes/c*.png'))
print('startframes:', len(made))
assert made, 'no startframes produced'

In [ ]:
# 8. Package only the stills
import os, shutil, zipfile
dest = '/kaggle/working/startframes'
if os.path.isdir('startframes'):
    shutil.copytree('startframes', dest, dirs_exist_ok=True)
ZIP = '/kaggle/working/stills.zip'
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as zf:
    for name in sorted(os.listdir(dest)):
        if name.endswith('.png'):
            zf.write(os.path.join(dest,name), name)
print('zip', os.path.getsize(ZIP), 'bytes')